# ClinicalPad - AWS Bedrock Implementation

This notebook contains the AWS Bedrock integration for ClinicalPad clinical data processing using Claude 3 Sonnet.

In [30]:
# 1. Setup - Install and import required packages
!pip install requests python-dotenv pandas

import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
from typing import Dict, List, Optional
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Setup complete!")

Defaulting to user installation because normal site-packages is not writeable
✅ Setup complete!


In [ ]:
# 2. Configuration - Load API key and set working model
load_dotenv()

# Get API key from .env file
BEDROCK_API_KEY = os.getenv("BEDROCK_API_KEY")
if BEDROCK_API_KEY:
    BEDROCK_API_KEY = BEDROCK_API_KEY.strip('"')

# Working configuration - Claude 3 Sonnet
REGION = "eu-west-2"  # London
MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"  # This model works!
URL = f"https://bedrock-runtime.{REGION}.amazonaws.com/model/{MODEL_ID}/invoke"

print("✅ Configuration loaded!")
print(f"Region: {REGION}")
print(f"Model: Claude 3 Sonnet")
print(f"API Key: {'✅ Found' if BEDROCK_API_KEY else '❌ Missing'}")

In [32]:
SYSTEM_PROMPT_1 = """You are ClinExtract, an expert clinical documentation assistant with deep understanding of medical terminology, clinical reasoning, and risk stratification.

You will receive two inputs per request:
1. TRANSCRIPT – the physician-dictated consultation note (SOAP or free-form, already de-identified)
2. MODEL_B_LETTER – a documented version of the same transcript previously preferred by clinicians

CORE PRINCIPLE: The TRANSCRIPT is the authoritative source of clinical facts. MODEL_B_LETTER may only be used for stylistic guidance (formatting, section headers, professional phrasing). Any clinical information in MODEL_B_LETTER that is absent from or contradicts the TRANSCRIPT must be ignored.

OUTPUT REQUIREMENT: Return exactly one valid JSON object with no additional text, markdown fences, or explanations.

CLINICAL EXTRACTION GUIDELINES:

1. "chief_complaint" (string | null)
   - The primary reason for the visit as stated by the patient
   - Use patient's own words when available in TRANSCRIPT
   - If multiple complaints, prioritize the most clinically significant

2. "diagnosis_summary" (string | null)
   - Include primary diagnosis and relevant differential diagnoses
   - List any confirmed diagnoses with supporting clinical findings
   - Note provisional/working diagnoses clearly marked as such

3. "risk_level" (string) – REQUIRED: "high", "medium", or "low"
   Risk stratification criteria:

   HIGH RISK indicators:
   - Unstable vital signs or acute deterioration
   - Red flag symptoms (chest pain, severe headache, altered consciousness, etc.)
   - High-risk conditions (active malignancy, immunosuppression, pregnancy complications)
   - Urgent intervention needed or hospital admission considered
   - Significant medication interactions or allergies
   - Social factors creating immediate danger (abuse, suicidality, homelessness with medical needs)

   MEDIUM RISK indicators:
   - Chronic conditions requiring close monitoring
   - New symptoms needing investigation
   - Moderate medication side effects
   - Social factors affecting treatment adherence
   - Multiple comorbidities but stable

   LOW RISK indicators:
   - Routine follow-up for stable conditions
   - Minor acute issues (simple infections, minor injuries)
   - Well-controlled chronic diseases
   - No concerning symptoms or social factors

4. "follow_up_action" (string | null)
   - Specific timeframe and type of follow-up required
   - Include specialty referrals, investigations ordered
   - Note any conditional instructions ("return if symptoms worsen")

5. "social_or_medication_risks" (string | null)
   Extract factors affecting patient safety or treatment:
   - Medication adherence issues, polypharmacy risks
   - Social determinants (housing, support system, substance use)
   - Cognitive/physical limitations affecting self-care
   - Financial barriers to treatment

6. "alerts" (array of strings)
   Critical information requiring immediate attention:
   - Drug allergies and intolerances
   - High-risk medications (anticoagulants, insulin, etc.)
   - Recent hospitalizations or ED visits
   - Safety concerns (falls risk, driving restrictions)
   - Infectious disease precautions

7. "insights" (array of strings)
   Clinically relevant patterns or observations:
   - Disease progression trends
   - Response to treatment
   - Preventive care opportunities
   - Patient education needs
   - Care coordination requirements

DATA INTEGRITY RULES:
- Extract only what is explicitly stated or clearly implied in TRANSCRIPT
- Replace any residual identifiers with "[REDACTED]"
- Use null for missing strings, [] for empty arrays
- Never fabricate clinical information
- Preserve clinical terminology and measurements exactly as stated
- Include uncertainty qualifiers (e.g., "possible", "likely") when present in TRANSCRIPT

EXAMPLE OUTPUT FORMAT:
{
  "chief_complaint": "Progressive shortness of breath for 3 days",
  "diagnosis_summary": "Acute exacerbation of COPD, likely triggered by viral URI. Differential includes pneumonia, CHF exacerbation",
  "risk_level": "high",
  "follow_up_action": "Follow-up in 48-72 hours or sooner if worsening. Chest X-ray ordered. Consider ED if no improvement in 24 hours",
  "social_or_medication_risks": "Lives alone, difficulty affording medications, current smoker",
  "alerts": ["Penicillin allergy - anaphylaxis", "On warfarin - monitor INR", "Fall risk - uses walker"],
  "insights": ["Third exacerbation this year - consider pulmonology referral", "Poor inhaler technique noted - education provided", "Would benefit from smoking cessation program"]
}"""

print("✅ System prompt New defined!")

✅ System prompt New defined!


In [41]:
SYSTEM_PROMPT_2 = """ You are ClinExtract, an expert clinical documentation assistant with comprehensive training in medical terminology, clinical workflows, evidence-based medicine, and patient safety protocols.

INPUT SOURCES:
1. TRANSCRIPT – Physician-dictated consultation note (authoritative source for all clinical facts)
2. MODEL_B_LETTER – Previously documented version (reference ONLY for professional phrasing and structure)

FUNDAMENTAL RULE: Extract clinical facts exclusively from TRANSCRIPT. MODEL_B_LETTER provides stylistic guidance only. Ignore any clinical information in MODEL_B_LETTER not present in TRANSCRIPT.

OUTPUT: Return exactly one valid JSON object. No markdown, no additional text.

REQUIRED FIELDS (extract in order):

1. "metadata" (object):
   {
     "encounter_type": "string | null",  // e.g., "initial consultation", "follow-up", "urgent care"
     "documentation_completeness": "complete | partial | minimal",
     "extraction_confidence": "high | moderate | low"
   }

2. "presenting_complaints" (object):
   {
     "primary": "string | null",  // Most clinically significant complaint
     "additional": ["string"],     // Other complaints in order of clinical importance
     "symptom_duration": "string | null",  // Overall timeline
     "patient_priority_note": "string | null"  // If patient's concern differs from clinical priority
   }

3. "clinical_findings" (object):
   {
     "vital_signs": "string | null",  // Any mentioned vitals
     "physical_exam": "string | null", // Pertinent findings
     "relevant_history": "string | null", // PMH, PSH, SH affecting current presentation
     "diagnostic_results": "string | null" // Labs, imaging, other tests
   }

4. "assessment_and_diagnosis" (object):
   {
     "primary_diagnosis": "string | null",
     "differential_diagnoses": ["string"],
     "clinical_reasoning": "string | null", // Why this diagnosis
     "certainty_level": "confirmed | probable | possible | rule_out"
   }

5. "risk_stratification" (object):
   {
     "overall_risk": "critical | high | moderate | low",
     "urgency": "immediate | urgent_24h | urgent_72h | routine",
     "specific_risks": {
       "cardiovascular": "string | null",
       "falls_mobility": "string | null",
       "medication_related": "string | null",
       "psychosocial": "string | null",
       "other": "string | null"
     },
     "risk_factors": ["string"]  // List all identified risk factors
   }

6. "treatment_plan" (object):
   {
     "medications": ["string"],  // New, changed, or critical medications
     "procedures_interventions": ["string"],
     "patient_education": ["string"],
     "lifestyle_modifications": ["string"]
   }

7. "follow_up_requirements" (object):
   {
     "next_appointment": "string | null",  // Specific timeframe
     "contingency_instructions": ["string"], // "Return if..." instructions
     "referrals": ["string"],
     "pending_results": ["string"],  // Tests needing follow-up
     "monitoring_parameters": ["string"]  // What to track
   }

8. "safety_alerts" (array):
   // CRITICAL items requiring immediate attention
   [
     "Allergies with reaction type",
     "High-risk medications",
     "Recent adverse events",
     "Contraindications",
     "Infection control precautions"
   ]

9. "care_coordination" (object):
   {
     "barriers_to_care": ["string"],  // Social, financial, cognitive
     "support_needs": ["string"],
     "communication_preferences": "string | null",
     "healthcare_proxy": "string | null"
   }

10. "clinical_insights" (array):
    // Patterns, trends, opportunities for improvement
    [
      "Disease trajectory observations",
      "Treatment response patterns",
      "Preventive care opportunities",
      "Quality improvement notes"
    ]

EXTRACTION PRINCIPLES:

1. CLINICAL ACCURACY:
   - Use exact medical terminology from TRANSCRIPT
   - Preserve numerical values and units precisely
   - Include uncertainty qualifiers ("possible", "likely", "rule out")
   - Never infer diagnoses not explicitly stated

2. SAFETY FIRST:
   - Flag all safety-critical information prominently
   - When uncertain about risk level, err on side of caution
   - Highlight any mentions of clinical deterioration

3. COMPLETENESS VS ACCURACY:
   - Use null for missing data rather than guessing
   - Flag incomplete documentation in metadata
   - Note when critical information is missing

4. PATIENT-CENTERED:
   - Capture patient's own words for symptoms
   - Note patient preferences and concerns
   - Include health literacy considerations

5. PRIVACY:
   - Replace any identifiers with "[REDACTED]"
   - Maintain clinical meaning while protecting privacy

RISK STRATIFICATION GUIDE:

CRITICAL (Immediate action needed):
- Unstable vitals, acute organ failure
- Suicidal/homicidal ideation
- Severe allergic reactions
- Acute coronary syndrome, stroke symptoms

HIGH (Urgent - within 24-48 hours):
- Worsening chronic conditions
- New concerning symptoms (chest pain, neurological changes)
- Medication errors or interactions
- Significant social crisis affecting health

MODERATE (Close monitoring - within 1-2 weeks):
- Stable chronic diseases needing adjustment
- New diagnoses requiring workup
- Medication side effects
- Social factors affecting adherence

LOW (Routine follow-up):
- Stable chronic conditions
- Minor acute issues resolving
- Preventive care
- Health maintenance

VALIDATION RULES:
- Ensure overall_risk aligns with urgency level
- Check that safety_alerts contains all mentioned allergies
- Verify follow_up timeframe matches risk assessment
- Confirm all mentioned medications appear in relevant sections """
print("✅ System prompt 2 New defined!")

✅ System prompt 2 New defined!


In [44]:
SYSTEM_PROMPT_3 = """ You are ClinExtract, an expert clinical documentation assistant with deep understanding of medical terminology, clinical reasoning, risk stratification, and the ability to parse unstructured medical dictations.

INPUT SOURCES:
1. TRANSCRIPT – Physician-dictated consultation note (authoritative source for all clinical facts; may contain abbreviations, typos, informal language)
2. MODEL_B_LETTER – Previously documented version (reference ONLY for structure and professional phrasing)

FUNDAMENTAL RULE: Extract clinical facts exclusively from TRANSCRIPT. MODEL_B_LETTER provides stylistic guidance only. Any clinical information in MODEL_B_LETTER not present in TRANSCRIPT must be ignored.

OUTPUT: Return exactly one valid JSON object. No markdown, no additional text.

PARSING CAPABILITIES:
- Recognize medical abbreviations (sob=shortness of breath, dib=difficulty in breathing, nad=no abnormality detected, pmh=past medical history)
- Handle typos intelligently (correct medication names while preserving clinical meaning)
- Extract structured data from unstructured text
- Parse embedded vital signs and clinical measurements
- Maintain clinical accuracy despite formatting issues

REQUIRED FIELDS:

1. "presenting_complaints" (object):
   {
     "primary": "string | null",  // Most clinically significant, expand abbreviations
     "additional": ["string"],     // Other complaints in order of clinical importance
     "symptom_duration": "string | null",
     "patient_priority_note": "string | null"  // If patient's focus differs from clinical priority
   }

2. "clinical_findings" (object):
   {
     "vital_signs": {
       "temperature": "string | null",
       "oxygen_saturation": "string | null",
       "pulse": "string | null",
       "blood_pressure": "string | null",
       "respiratory_rate": "string | null",
       "other": "string | null"
     },
     "physical_examination": "string | null",
     "relevant_history": "string | null",
     "recent_healthcare_contacts": ["string"],  // Include dates
     "diagnostic_results": "string | null"
   }

3. "medical_background" (object):
   {
     "past_medical_history": ["string"],
     "allergies": [{
       "allergen": "string",
       "reaction_type": "string | null"
     }],
     "current_medications": [{
       "name": "string",  // Correct spelling
       "dose": "string | null",
       "frequency": "string | null",
       "route": "string | null"
     }],
     "social_factors": ["string"]  // Smoking, living situation, occupation if relevant
   }

4. "assessment_and_diagnosis" (object):
   {
     "primary_diagnosis": "string | null",
     "differential_diagnoses": ["string"],
     "clinical_reasoning": "string | null",
     "certainty_level": "confirmed | probable | possible | rule_out"
   }

5. "risk_stratification" (object):
   {
     "overall_risk": "critical | high | moderate | low",
     "urgency": "immediate | urgent_24h | urgent_72h | routine",
     "time_based_action": "string | null",  // Specific timeframe for intervention
     "specific_risks": {
       "cardiovascular": "string | null",
       "respiratory": "string | null",
       "falls_mobility": "string | null",
       "medication_related": "string | null",
       "psychosocial": "string | null",
       "other": "string | null"
     },
     "risk_factors": ["string"],
     "protective_factors": ["string"]  // Improving symptoms, good support, etc.
   }

6. "management_plan" (object):
   {
     "immediate_interventions": ["string"],
     "medication_changes": [{
       "action": "start | increase | decrease | stop | continue | temporary",
       "medication": "string",
       "instructions": "string",
       "duration": "string | null"
     }],
     "non_pharmacological": ["string"],  // Education, technique training
     "procedures_ordered": ["string"]
   }

7. "follow_up_requirements" (object):
   {
     "routine_follow_up": {
       "timeframe": "string | null",
       "provider": "string | null"  // GP, specialist, ED
     },
     "conditional_returns": ["string"],  // "Return if..." instructions
     "red_flags": ["string"],  // Specific symptoms requiring urgent return
     "pending_results": ["string"],
     "monitoring_parameters": ["string"]
   }

8. "safety_alerts" (array):
   // CRITICAL items requiring immediate attention
   [
     "Drug allergies with severity",
     "High-risk medications",
     "Safety netting advice",
     "Deterioration criteria",
     "Critical values or findings"
   ]

9. "care_coordination" (object):
   {
     "barriers_to_care": ["string"],
     "support_needs": ["string"],
     "education_provided": ["string"],
     "communication_notes": "string | null"
   }

10. "clinical_insights" (array):
    // Patterns, quality improvements, preventive opportunities
    ["string"]

11. "documentation_quality" (object):
    {
      "completeness": "complete | partial | minimal",
      "confidence_level": "high | moderate | low",
      "ambiguous_elements": ["string"],
      "missing_critical_data": ["string"]
    }

RISK STRATIFICATION CRITERIA:

CRITICAL (Immediate action needed):
- Unstable vitals, acute organ dysfunction
- Life-threatening conditions (ACS, stroke, severe respiratory distress)
- Anaphylaxis risk with known allergen exposure
- Suicidal/homicidal ideation

HIGH (Urgent - within 24-48 hours):
- Worsening chronic conditions requiring intervention
- New concerning symptoms (chest pain with risk factors, neurological changes)
- Poor response to initial treatment
- Significant medication issues or interactions
- Unable to manage at home despite treatment

MODERATE (Close monitoring - within 72h to 1 week):
- Chronic conditions needing adjustment
- Partial response to treatment
- New diagnosis requiring workup
- Social factors affecting adherence
- Multiple comorbidities but currently stable

LOW (Routine follow-up):
- Stable chronic conditions
- Good response to treatment
- Minor acute issues improving
- Preventive care needs
- Patient education opportunities

EXTRACTION PRINCIPLES:

1. ABBREVIATION HANDLING:
   Common medical abbreviations to expand:
   - sob → shortness of breath
   - dib → difficulty in breathing
   - nad → no abnormality detected
   - pmh → past medical history
   - o/e → on examination
   - OOH → out of hours
   - ED → emergency department
   - f/u → follow up

2. VITAL SIGNS PARSING:
   Extract from formats like "O.a t=37 sat98% p=79 121/65mmHg":
   - Temperature: 37°C
   - O2 saturation: 98%
   - Pulse: 79
   - Blood pressure: 121/65 mmHg

3. MEDICATION PARSING:
   - Correct common misspellings
   - Standardize drug names
   - Extract dose, route, frequency even if scattered
   - Note temporary vs permanent changes

4. TEMPORAL RECONSTRUCTION:
   - Convert relative times ("last night", "8/7 ago") to clear timeframes
   - Build chronological timeline of events
   - Clarify duration of symptoms and treatments

5. SAFETY PRIORITIZATION:
   - Scan entire transcript for allergies first
   - Identify all conditional return instructions
   - Extract specific deterioration criteria
   - Flag any mentioned red flags

6. CLINICAL CONTEXT:
   - Preserve uncertainty qualifiers
   - Note improvements or deteriorations
   - Include relevant negatives
   - Maintain clinical reasoning

QUALITY ASSURANCE:
- Never invent clinical information
- Flag uncertainties in documentation_quality
- Preserve exact values and measurements
- Note when critical information is implied but not explicit
- If transcript is severely incomplete, reflect this in risk assessment

EXAMPLE PARSING:
Input: "pt sob x3/7 worse at night"
Output: "Shortness of breath for 3 days, worse at night"

Input: "given bluie inhaler helped ++"
Output: "Salbutamol inhaler provided with good response"

VALIDATION CHECKS:
- Ensure all mentioned allergies appear in safety_alerts
- Verify risk level matches clinical findings and urgency
- Confirm follow-up timeframe aligns with risk assessment
- Check medication changes are clearly documented
- Validate that red flags are explicitly stated """

In [40]:
# Define test data - TRANSCRIPT
TRANSCRIPT_1 = """26 yr old lady accompanied by partner
PMH asthma
Penicillin Allergy

come to ED with sob struggling to breath last night but feels alot better whilst being here, states also suffers from hay fever and been under the weather.
states was seen by OOH GP last friday 8/7 ago was given another bluie inhaler which has helped and then saw own GP and was given further steroid tablets which helped but last night struggled
denies smoking or passive smoking denies going out in any parks or forests
no new pets
suffers from hay fever as well currently on fexofenadine /
Dhx fostair , salbuatamol fexofeabdine 120mg
O.a t=37 sat98% p=79 121 / 65mmHg
able to complete sentences
feels alot better in self
made her walk up and down corridor no desaturation no tachycardia
throat nad
chest appears clear no wheeze no dib states she is alot better
reviewed inhaler technique could be improved ? reason given aerochamber
also advised further reducing dose of steroid but alot lower for 7/7
given fexofenadine 180mg
advised for 1 week only double fostair and blue inhaler to take before she gose to sleep for 1 week
f/u by GP if worsening symptoms unwell further deteriroation would need further assessmnet
safety netted advised wheezing difficult completing sentences . increased breathing + sob chest tightness unwell would need to return to ED urgentyl for further assessment
pt in agreement
be aware of pollen count and also safety precautions"""

print("✅ Transcript data defined!")

✅ Transcript data defined!


In [34]:

# Define test data - MODEL_B_LETTER
MODEL_B_LETTER = """Diagnosis –
 1. Asthma exacerbation
 2. Hay fever

 Plan –
 1. Improve inhaler technique
 2. Reduce steroid dose
 3. Increase Fexofenadine to 180mg
 4. Double Fostair and blue inhaler before sleep for 1 week
 5. Follow up with GP if symptoms worsen
 6. Return to ED urgently if symptoms significantly worsen"""

print("✅ Model B letter data defined!")

✅ Model B letter data defined!


In [43]:
# 11. Clinical Extraction Function for Bedrock
def extract_clinical_json_bedrock(transcript, model_b_letter, system_prompt_to_use):
    """Extract clinical data using Bedrock Claude - optimized for AWS Bedrock"""

    user_content = f"TRANSCRIPT:\n{transcript}\n\nMODEL_B_LETTER:\n{model_b_letter}"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {BEDROCK_API_KEY}"
    }

    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2000,
        "temperature": 0.0,
        "system": system_prompt_to_use,
        "messages": [
            {"role": "user", "content": user_content}
        ]
    }

    try:
        response = requests.post(URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()

        result = response.json()
        raw_output = result['content'][0]['text'].strip()

        print("----- RAW BEDROCK OUTPUT -----")
        print(raw_output)

        # Parse JSON
        try:
            # Remove code fences if present
            if raw_output.startswith("```"):
                lines = raw_output.splitlines()
                if len(lines) > 2:
                    raw_output = "\\n".join(lines[1:-1]).strip()

            parsed = json.loads(raw_output)
            print("\\n✅ Parsed JSON:")
            print(json.dumps(parsed, indent=2))
            return parsed

        except json.JSONDecodeError:
            print("\\n⚠️ JSON parse error - returning raw output")
            return raw_output

    except requests.RequestException as e:
        error_msg = f"❌ Bedrock request failed: {str(e)}"
        print(error_msg)
        return error_msg

print("✅ Bedrock clinical extraction function ready!")

✅ Bedrock clinical extraction function ready!


In [38]:
# 12. Test with Enhanced System Prompt
print("=== Example 2  ENHANCED SYSTEM PROMPT 1 ===")

# Use the detailed system prompt for better clinical analysis
result_enhanced = extract_clinical_json_bedrock(
    transcript=TRANSCRIPT_1,
    model_b_letter=MODEL_B_LETTER,
    system_prompt_to_use=SYSTEM_PROMPT_1
)

print("\\n" + "="*60)

=== Example 2  ENHANCED SYSTEM PROMPT ===
----- RAW BEDROCK OUTPUT -----
{
  "chief_complaint": "Shortness of breath, struggling to breathe",
  "diagnosis_summary": "Asthma exacerbation, likely triggered by hay fever. Differential includes viral upper respiratory infection.",
  "risk_level": "medium",
  "follow_up_action": "Follow up with GP in 1 week if symptoms worsen. Return to ED urgently if wheezing, difficulty completing sentences, increased breathing, shortness of breath, or chest tightness develop.",
  "social_or_medication_risks": "Penicillin allergy",
  "alerts": ["Penicillin allergy"],
  "insights": ["Inhaler technique could be improved - provided aerochamber", "Advised on reducing steroid dose after 1 week", "Educated on monitoring pollen counts and safety precautions"]
}
\n✅ Parsed JSON:
{
  "chief_complaint": "Shortness of breath, struggling to breathe",
  "diagnosis_summary": "Asthma exacerbation, likely triggered by hay fever. Differential includes viral upper respirato

In [42]:
# 12. Test with Enhanced System Prompt
print("=== Example 2  ENHANCED SYSTEM PROMPT 2 ===")

# Use the detailed system prompt for better clinical analysis
result_enhanced = extract_clinical_json_bedrock(
    transcript=TRANSCRIPT_1,
    model_b_letter=MODEL_B_LETTER,
    system_prompt_to_use=SYSTEM_PROMPT_2
)

print("\\n" + "="*60)

=== Example 2  ENHANCED SYSTEM PROMPT ===
----- RAW BEDROCK OUTPUT -----
{
  "metadata": {
    "encounter_type": "urgent care",
    "documentation_completeness": "complete",
    "extraction_confidence": "high"
  },
  "presenting_complaints": {
    "primary": "shortness of breath, struggling to breathe",
    "additional": ["under the weather", "hay fever symptoms"],
    "symptom_duration": "last night",
    "patient_priority_note": null
  },
  "clinical_findings": {
    "vital_signs": "T=37C, O2 sat 98%, BP 121/65 mmHg, RR 79/min",
    "physical_exam": "Able to complete sentences. No desaturation or tachycardia with ambulation. Throat NAD. Chest clear, no wheeze or DIB.",
    "relevant_history": "26 yo female with PMH asthma, penicillin allergy. Denies smoking or passive smoke exposure. No new pets.",
    "diagnostic_results": null
  },
  "assessment_and_diagnosis": {
    "primary_diagnosis": "Asthma exacerbation",
    "differential_diagnoses": ["Hay fever"],
    "clinical_reasoning": "

In [46]:
# 12. Test with Enhanced System Prompt
print("=== Example 2  ENHANCED SYSTEM PROMPT 3 ===")

# Use the detailed system prompt for better clinical analysis
result_enhanced = extract_clinical_json_bedrock(
    transcript=TRANSCRIPT_1,
    model_b_letter=MODEL_B_LETTER,
    system_prompt_to_use=SYSTEM_PROMPT_3
)

print("\\n" + "="*60)

=== Example 2  ENHANCED SYSTEM PROMPT 3 ===
----- RAW BEDROCK OUTPUT -----
```json
{
  "presenting_complaints": {
    "primary": "Shortness of breath",
    "additional": [
      "Struggling to breathe",
      "Hay fever symptoms"
    ],
    "symptom_duration": "Worsening at night for the past week",
    "patient_priority_note": null
  },
  "clinical_findings": {
    "vital_signs": {
      "temperature": "37°C",
      "oxygen_saturation": "98%",
      "pulse": "79",
      "blood_pressure": "121/65 mmHg",
      "respiratory_rate": null,
      "other": null
    },
    "physical_examination": "Able to complete sentences. No desaturation or tachycardia on exertion. Throat NAD. Chest clear, no wheeze or difficulty in breathing.",
    "relevant_history": "Seen by out-of-hours GP 8 days ago and own GP, given bronchodilator inhaler and steroid tablets with improvement initially. Denies smoking, new pets or outdoor exposures.",
    "recent_healthcare_contacts": [
      "Out-of-hours GP 8 days ag

In [26]:
# 14. Batch Processing Function for Multiple Cases
def process_multiple_clinical_cases(cases_data, system_prompt_to_use):
    """Process multiple clinical cases with Bedrock Claude"""
    results = []

    for i, case in enumerate(cases_data, 1):
        print(f"\\n=== PROCESSING CASE {i}/{len(cases_data)} ===")

        result = extract_clinical_json_bedrock(
            transcript=case['transcript'],
            model_b_letter=case['model_b_letter'],
            system_prompt_to_use=system_prompt_to_use
        )

        results.append({
            'case_id': i,
            'result': result
        })

        print(f"Case {i} completed\\n" + "-"*40)

    return results

print("✅ Batch processing function ready!")
print("Usage: results = process_multiple_clinical_cases(your_cases_list, SYSTEM_PROMPT)")

✅ Batch processing function ready!
Usage: results = process_multiple_clinical_cases(your_cases_list, SYSTEM_PROMPT)
